# AutoGluon Modeling (Competition Optimized)

This notebook trains AutoGluon models using a configuration strictly optimized for the competition's data structure and metric.

## Key Configuration
- **Data Source**: Switchable between Raw and Augmented features.
- **Leakage Prevention**: Uses `groups='survey_id'` with `num_bag_folds=3`. 
  - *Why 3?* There are exactly 3 surveys (100k, 200k, 300k). Validating on an entire survey requires 3 folds (Leave-One-Group-Out). Using more folds is impossible with distinct groups.
- **Metric**: **EXACT** competition metric (10% Consumption MAPE + 90% Weighted Poverty Rates), implemented 1:1 with official rules.

In [ ]:
import pandas as pd
import numpy as np
from autogluon.tabular import TabularPredictor
from autogluon.core.metrics import make_scorer

## 1. Configuration & Data Loading

In [ ]:
# === CONFIGURATION ===
USE_AUGMENTED_DATA = False
AUGMENTED_PATH = 'data/train_features_extreme_custom.csv'
RAW_PATH = 'data/train_hh_features.csv'
GT_PATH = 'data/train_hh_gt.csv'
# =====================

def load_data(use_augmented):
    gt_df = pd.read_csv(GT_PATH)
    
    if use_augmented:
        print(f"Loading Augmented Data from: {AUGMENTED_PATH}")
        features_df = pd.read_csv(AUGMENTED_PATH)
    else:
        print(f"Loading Raw Data from: {RAW_PATH}")
        features_df = pd.read_csv(RAW_PATH)
        
    needed_cols = ['cons_ppp17', 'survey_id']
    cols_to_merge = [c for c in needed_cols if c not in features_df.columns]
    
    if cols_to_merge:
        if 'hhid' not in features_df.columns and 'hhid' in gt_df.columns:
            raise ValueError("features_df missing 'hhid' for merge.")
        merged_df = pd.merge(features_df, gt_df[['hhid'] + cols_to_merge], on='hhid', how='left')
        # Ensure original index is preserved or reset cleanly for later lookup
        merged_df = merged_df.reset_index(drop=True)
        return merged_df
    else:
        features_df = features_df.reset_index(drop=True)
        return features_df

train_data = load_data(USE_AUGMENTED_DATA)

# Setup global lookup for survey_id (Critical for Metric)
SURVEY_ID_LOOKUP = train_data['survey_id'].copy()
print(f"Data Shape: {train_data.shape}")

## 2. Exact Competition Metric Implementation
Implemented 1:1 with the provided specifications:
- Linear weights centered at 0.4
- Thresholds derived from Survey 300000
- Per-Survey aggregation

In [ ]:
import numpy as np

class CompetitionMetricExact:
    def __init__(self, eps: float = 1e-8):
        self.eps = float(eps)
        self.percentiles = np.arange(0.05, 1.0, 0.05)
        self.weights = 1.0 - np.abs(0.4 - self.percentiles)
        self.sum_w = float(np.sum(self.weights))
        self.thresholds = None

    def fit_thresholds_from_training(self, y_train_consumption, train_survey_ids, threshold_survey_id=300000):
        y_train_consumption = np.asarray(y_train_consumption, dtype=float)
        train_survey_ids = np.asarray(train_survey_ids)
        mask = (train_survey_ids == threshold_survey_id)
        vals = y_train_consumption[mask]
        if vals.size == 0:
            raise ValueError(f"No training rows found for survey_id={threshold_survey_id}.")
        self.thresholds = np.quantile(vals, self.percentiles)
        return self.thresholds

    def score_vectorized(self, y_true, y_pred, survey_ids):
        if self.thresholds is None:
            raise ValueError("Thresholds not set.")

        yt = np.asarray(y_true, dtype=float)
        yp = np.asarray(y_pred, dtype=float)
        sid = np.asarray(survey_ids)

        n = yt.size
        if n == 0:
            return float("nan")
        if yp.size != n or sid.size != n:
            raise ValueError("y_true, y_pred, survey_ids must have the same length.")

        # Sort by survey id so groups are contiguous
        order = np.argsort(sid, kind="mergesort")
        sid_s = sid[order]
        yt_s = yt[order]
        yp_s = yp[order]

        # Group boundaries
        change = np.empty(n, dtype=bool)
        change[0] = True
        change[1:] = sid_s[1:] != sid_s[:-1]
        starts = np.flatnonzero(change)
        counts = np.diff(np.append(starts, n))
        S = starts.size

        eps = self.eps

        # Consumption MAPE per survey
        denom_cons = np.maximum(np.abs(yt_s), eps)
        cons_term = np.abs(yt_s - yp_s) / denom_cons
        cons_sum = np.add.reduceat(cons_term, starts)
        cons_mape = cons_sum / counts

        # Poverty rates per survey per threshold
        thr = np.asarray(self.thresholds, dtype=float)  # (T,)
        true_below = (yt_s[:, None] < thr[None, :])
        pred_below = (yp_s[:, None] < thr[None, :])

        true_cnt = np.add.reduceat(true_below, starts, axis=0)  # (S,T)
        pred_cnt = np.add.reduceat(pred_below, starts, axis=0)  # (S,T)

        true_rates = true_cnt / counts[:, None]
        pred_rates = pred_cnt / counts[:, None]

        denom_rates = np.maximum(np.abs(true_rates), eps)
        rate_mape = np.abs(pred_rates - true_rates) / denom_rates  # (S,T)

        w = self.weights.astype(float)
        pov_part = (90.0 / self.sum_w) * (rate_mape * w[None, :]).sum(axis=1)
        cons_part = 10.0 * cons_mape

        return float((pov_part + cons_part).mean())

    # Optional: keep original name 'score' but now vectorized
    def score(self, y_true, y_pred, survey_ids):
        return self.score_vectorized(y_true, y_pred, survey_ids)


# Initialize and Fit Thresholds
metric_obj = CompetitionMetricExact()
metric_obj.fit_thresholds_from_training(
    train_data['cons_ppp17'].values,
    train_data['survey_id'].values
)
print("Thresholds derived:", metric_obj.thresholds)


# Wrapper for AutoGluon
def ag_score_wrapper(y_true, y_pred):
    # Attempt to retrieve survey_ids via index
    s_ids = SURVEY_ID_LOOKUP.loc[y_true.index].values
    # now uses vectorized scoring (math identical)
    return metric_obj.score_vectorized(y_true, y_pred, s_ids)


ag_scorer = make_scorer(
    name='competition_loss_exact',
    score_func=ag_score_wrapper,
    optimum=0,
    greater_is_better=False
)

## 3. Train AutoGluon Predictor

In [ ]:
label = 'cons_ppp17'
save_path = 'ag_models_competition_exact'

predictor = TabularPredictor(
    label=label, 
    eval_metric=ag_scorer,
    path=save_path,
    groups='survey_id'
)

predictor.fit(
    train_data,
    presets='best_quality',
    groups='survey_id',
    num_bag_folds=3,
    time_limit=3600*2
)

In [ ]:
predictor.leaderboard()